# 2608.27372 — Ellipsoid-Fitting Phase Boundary

**Engineering Statements companion notebook**

This notebook treats the Engineering Statement YAML as the primary specification and adds an executable computational reading:

\[
\text{asymptotic specification}
\rightarrow
\text{computation}
\rightarrow
\text{reading}.
\]

Source: arXiv:2608.27372  
Statement path: `statements/2608-27372.yaml`

The paper studies random vectors \(x_1,\ldots,x_n\in\mathbb{R}^d\) and ellipsoid fitting through

\[
x_i^\top R x_i = 1
\qquad i=1,\ldots,n,
\]

with constraint density

\[
\alpha=\frac{n}{d^2}.
\]

The asymptotic SAT–UNSAT phase boundary is denoted \(\alpha_\star(\kappa)\), where

\[
\kappa=\mathbb{E}[x_{ij}^4].
\]

For Gaussian coordinates,

\[
\kappa=3,
\qquad
\alpha_\star(3)=\frac14.
\]

## 1. Install dependencies

The notebook uses:

- `pyyaml` to read the Engineering Statement
- `numpy` for random instances
- `cvxpy` to solve the ellipsoid-fitting semidefinite program
- `pandas` and `matplotlib` for readings and plots

In [ ]:
import sys
import subprocess
import importlib.util

packages = {
    "yaml": "pyyaml",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "cvxpy": "cvxpy",
}

missing = [pip_name for module, pip_name in packages.items()
           if importlib.util.find_spec(module) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

print("Dependencies ready.")

## 2. Load the Engineering Statement

The notebook expects the repository layout:

```text
engineering-statements/
├── statements/
│   └── 2608-27372.yaml
├── notebooks/
│   └── 2608-27372.ipynb
└── outputs/
    └── 2608-27372/
```

For portability, a source-native fallback statement is included if the YAML file is not found.

In [ ]:
from pathlib import Path
import yaml

STATEMENT_PATH = Path("../statements/2608-27372.yaml")

fallback_yaml = """
id: 2608-27372
title: Universality and Sharp Thresholds for Ellipsoid Fitting

source:
  paper: https://arxiv.org/abs/2608.27372
  lab_report: https://labreports.app/2608-27372/
  repo: https://github.com/thinkthoughts/engineering-statements

objective: >
  Specify the ellipsoid-fitting problem from its random-vector input
  through its constraint-density scaling, sharp SAT-UNSAT phase boundary,
  distribution-dependent threshold, and computational readings.

contexts:
  - ellipsoid fitting
  - high-dimensional probability
  - random constraint-satisfaction problems
  - convex geometry
  - semidefinite programming
  - asymptotic phase transitions
  - computational readings

constraints:
  - Random vectors x_1,...,x_n lie in R^d.
  - Seek a positive-semidefinite matrix R such that x_i^T R x_i = 1 for every i.
  - Constraint density is specified by alpha = n / d^2.
  - The asymptotic regime has n,d increasing with n / d^2 approaching alpha.
  - A sharp SAT-UNSAT phase boundary occurs at alpha_star(kappa).
  - Below alpha_star(kappa), ellipsoid fitting is feasible with high probability.
  - Above alpha_star(kappa), ellipsoid fitting is infeasible with high probability.
  - The coordinate distribution enters the phase boundary through kappa = E[x_ij^4].
  - alpha_star(kappa) is nonincreasing in kappa.
  - For Gaussian coordinates, kappa = 3 and alpha_star(3) = 1/4.
  - The Gaussian phase boundary therefore occurs asymptotically near n = d^2 / 4.
  - Computational readings at a chosen dimension are distinguished from the asymptotic specification.

outputs:
  - engineering statement YAML
  - companion notebook
  - reproducible SDP experiment

next_steps:
  - reproduce the ellipsoid-fitting SDP
  - vary n to generate computational SAT-UNSAT readings
  - compare the observed transition with alpha_star(kappa)
  - reproduce the Gaussian reading kappa = 3 and alpha_star(3) = 1/4
  - test coordinate distributions with different values of kappa
"""

if STATEMENT_PATH.exists():
    statement = yaml.safe_load(STATEMENT_PATH.read_text())
    print(f"Loaded {STATEMENT_PATH}")
else:
    statement = yaml.safe_load(fallback_yaml)
    print("Statement file not found; using embedded fallback.")

statement

## 3. Render the statement

This keeps the YAML inspectable while exposing the pieces used by the computation.

In [ ]:
def render_statement(statement):
    print(f"# {statement['title']}")
    print(f"ID: {statement['id']}")
    print("\nObjective:")
    print(statement["objective"].strip())

    print("\nConstraints:")
    for item in statement.get("constraints", []):
        print(f"- {item}")

    print("\nNext steps:")
    for item in statement.get("next_steps", []):
        print(f"- {item}")

render_statement(statement)

## 4. Specify the Gaussian boundary

For Gaussian coordinates:

\[
\kappa=3
\]

and the theoretical phase boundary is

\[
\alpha_\star(3)=\frac14.
\]

At \(d=40\),

\[
n\approx \alpha_\star d^2
      = \frac14(40)^2
      = 400.
\]

In [ ]:
d_reference = 40
kappa_gaussian = 3.0
alpha_star_gaussian = 1.0 / 4.0
n_boundary_reference = alpha_star_gaussian * d_reference**2

{
    "d": d_reference,
    "kappa": kappa_gaussian,
    "alpha_star": alpha_star_gaussian,
    "predicted_n_boundary": n_boundary_reference,
}

## 5. Generate random coordinate instances

The default generator below uses standard Gaussian coordinates.

For each chosen \(d\) and \(n\), it returns a matrix whose rows are the vectors

\[
x_1,\ldots,x_n.
\]

In [ ]:
import numpy as np

def gaussian_instance(d, n, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    return rng.normal(size=(n, d))

# Lightweight example
rng = np.random.default_rng(260827372)
X = gaussian_instance(d=8, n=16, rng=rng)
X.shape

## 6. Solve the ellipsoid-fitting SDP

For one random instance, solve for a symmetric matrix \(R\) satisfying

\[
x_i^\top R x_i = 1
\qquad\text{for all } i,
\]

together with the positive-semidefinite matrix constraint.

The result is recorded as a computational reading:

```text
SAT
```

or

```text
UNSAT
```

In [ ]:
import cvxpy as cp

def ellipsoid_fit_reading(X, solver=None, verbose=False):
    X = np.asarray(X, dtype=float)
    n, d = X.shape

    R = cp.Variable((d, d), symmetric=True)

    constraints = [R >> 0]
    constraints += [
        cp.quad_form(X[i], R) == 1
        for i in range(n)
    ]

    problem = cp.Problem(cp.Minimize(0), constraints)

    installed = set(cp.installed_solvers())
    if solver is None:
        for candidate in ["CLARABEL", "SCS"]:
            if candidate in installed:
                solver = candidate
                break

    try:
        problem.solve(solver=solver, verbose=verbose)
    except Exception as exc:
        return {
            "reading": "ERROR",
            "status": type(exc).__name__,
            "solver": solver,
            "R": None,
        }

    feasible_statuses = {
        cp.OPTIMAL,
        cp.OPTIMAL_INACCURATE,
    }

    reading = "SAT" if problem.status in feasible_statuses else "UNSAT"

    return {
        "reading": reading,
        "status": problem.status,
        "solver": solver,
        "R": None if R.value is None else np.asarray(R.value),
    }

## 7. Lightweight computational reading

This small check verifies that the workflow executes. It is not intended to reproduce the asymptotic transition.

In [ ]:
rng = np.random.default_rng(260827372)
X_small = gaussian_instance(d=6, n=8, rng=rng)
small_reading = ellipsoid_fit_reading(X_small)

{
    "d": 6,
    "n": 8,
    "alpha": 8 / 6**2,
    "reading": small_reading["reading"],
    "status": small_reading["status"],
    "solver": small_reading["solver"],
}

## 8. Sweep constraint density

For a chosen dimension \(d\), vary \(n\), solve repeated random instances, and record

\[
\text{fraction SAT}
=
\frac{\text{SAT readings}}{\text{number of trials}}.
\]

This produces the computational transition that can be compared with the asymptotic specification.

In [ ]:
import pandas as pd

def sweep_gaussian(
    d,
    alphas,
    trials=8,
    seed=260827372,
    solver=None,
):
    rng = np.random.default_rng(seed)
    rows = []

    for alpha in alphas:
        n = max(1, int(round(alpha * d**2)))
        sat_count = 0
        statuses = []

        for trial in range(trials):
            X = gaussian_instance(d=d, n=n, rng=rng)
            result = ellipsoid_fit_reading(X, solver=solver)
            statuses.append(result["status"])
            sat_count += int(result["reading"] == "SAT")

        rows.append({
            "d": d,
            "n": n,
            "alpha": n / d**2,
            "trials": trials,
            "sat_count": sat_count,
            "fraction_sat": sat_count / trials,
            "statuses": statuses,
        })

    return pd.DataFrame(rows)

## 9. Paper-scale configuration

The paper's numerical comparison uses \(d=40\). The configuration below is ready to run, but is left unexecuted here because repeated SDP solves can take time.

A useful first sweep brackets the Gaussian theoretical boundary \(lpha_\star(3)=1/4\).

In [ ]:
# Paper-scale run:
#
# d = 40
# alphas = np.linspace(0.10, 0.40, 13)
# readings = sweep_gaussian(
#     d=d,
#     alphas=alphas,
#     trials=8,
#     seed=260827372,
# )
#
# readings

## 10. Plot computational readings against the specification

The vertical reference marks the Gaussian theoretical boundary. The plotted points come from computation and should be interpreted as readings relative to that specification.

In [ ]:
import matplotlib.pyplot as plt

def plot_readings(readings, alpha_star=1/4):
    fig, ax = plt.subplots(figsize=(8, 5))

    ax.plot(
        readings["alpha"],
        readings["fraction_sat"],
        marker="o",
        linewidth=1.5,
    )
    ax.axvline(alpha_star, linestyle="--")

    ax.set_xlabel(r"$\alpha=n/d^2$")
    ax.set_ylabel("fraction SAT")
    ax.set_ylim(-0.05, 1.05)
    ax.set_title("Computational readings relative to the phase boundary")

    plt.show()

# After running the paper-scale sweep:
# plot_readings(readings, alpha_star=alpha_star_gaussian)

## 11. Save reproducible outputs

Run this after computing `readings`.

In [ ]:
OUTPUT_DIR = Path("../outputs/2608-27372")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Example:
# readings.to_csv(OUTPUT_DIR / "gaussian_d40_readings.csv", index=False)
# print(OUTPUT_DIR / "gaussian_d40_readings.csv")

## 12. Engineering reading

The notebook keeps two objects separate:

\[
\boxed{\alpha_\star(\kappa)}
\]

is the **asymptotic specification**, while repeated SDP results at chosen \(d\) and \(n\) are **computational readings**.

For Gaussian coordinates:

\[
\boxed{
\kappa=3
\rightarrow
\alpha_\star(3)=\frac14
\rightarrow
d=40
\rightarrow
n\approx400
}
\]

The computational experiment can then ask how the observed SAT–UNSAT transition tracks that specified boundary.